# Parametric α-GRPO on Kaggle
Select GPU T4 ×2 and enable Internet. Point this notebook at a writable checkout. Run one stage at a time using the repository runbook.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/Parametric-alpha-GRPO')
ARTIFACTS = Path('/kaggle/working/alpha-grpo')

def launch(*args):
    subprocess.run(['bash', 'scripts/kaggle.sh', *map(str, args)], cwd=REPO, check=True)

Set `STAGE` to `setup`, `prepare`, `prescreen`, `checkpoint-gate`, `nccl-gate`, `smoke`, or `train`. Use a fresh run directory when changing the optimizer. Read `docs/kaggle.md` before selecting the synchronization backend.

In [ ]:
STAGE = ''
OPTIMIZER = 'gpu'
SYNC = 'auto'
RESUME = False
EXPERIMENT = 'control-a05'
SEED = 42
CHECKPOINT_ROOT = None  # Set a writable external path for this run if needed.

prepared = ARTIFACTS / 'prepared'
curated = ARTIFACTS / 'curated'
extra = ['--optimizer', OPTIMIZER]
if CHECKPOINT_ROOT is not None:
    extra += ['--checkpoint-root', CHECKPOINT_ROOT]
if RESUME:
    extra += ['--resume']

stages = {
    'setup': ['setup'],
    'prepare': ['prepare-data', '--out', prepared],
    'prescreen': ['prescreen', '--data', prepared, '--out', curated],
    'checkpoint-gate': ['preflight', '--data', curated, '--out', ARTIFACTS / f'checkpoint-gate-{OPTIMIZER}', '--sync', 'checkpoint', *extra],
    'nccl-gate': ['preflight', '--data', curated, '--out', ARTIFACTS / f'nccl-gate-{OPTIMIZER}', '--sync', 'nccl', *extra],
    'smoke': ['smoke', '--data', curated, '--out', ARTIFACTS / f'smoke-{OPTIMIZER}', '--sync', SYNC, *extra],
    'train': ['train', '--data', curated, '--out', ARTIFACTS / f'{EXPERIMENT}-{SEED}', '--sync', SYNC, '--experiment', EXPERIMENT, '--seed', SEED, *extra],
}

if STAGE:
    launch(*stages[STAGE])